In [ ]:
def reparer_total_charges(df):
    """
    Convertit TotalCharges en numérique et traite les trous révélés.
    Renvoie le dataframe réparé.
    """
    df = df.copy() 

    valeurs_uniques = df["TotalCharges"].unique()
    test_conversion = pd.to_numeric(df["TotalCharges"], errors="coerce")
    pct_nan_apres = test_conversion.isna().mean()

    if pct_nan_apres > 0.5:
        print("⛔ REFUS : plus de 50% de NaN après conversion, colonne probablement pas numérique.")
        print(f"   ({pct_nan_apres:.1%} de NaN détectés)")
        return df


    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    nb_trous = df["TotalCharges"].isna().sum()
    print(f"🔍 Trous démasqués dans TotalCharges : {nb_trous}")

    if nb_trous > 0:
      
        print("\n🔎 Détail des lignes concernées :")
        print(df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].to_string())

   
        mediane = df["TotalCharges"].median()
        df["TotalCharges"] = df["TotalCharges"].fillna(mediane)
        print(f"\n✅ Imputation par la médiane : {mediane:.2f}")
        print(f"   Trous restants : {df['TotalCharges'].isna().sum()}")

    print(f"\n📊 Nouveau type de TotalCharges : {df['TotalCharges'].dtype}")
    return df


df = reparer_total_charges(df)
print("\n✅ Phase 2 terminée.")

In [ ]:
print("Type TotalCharges :", df["TotalCharges"].dtype)
print("NaN restants :", df["TotalCharges"].isna().sum())
print("Aperçu :")
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

In [ ]:


def verifier_format_numerique(df, colonne):
    """
    Vérifie si une colonne texte contient des virgules comme séparateur décimal.
    Alerte si c'est le cas.
    """
    if df[colonne].dtype == object:
        masque_virgule = df[colonne].astype(str).str.contains(",", na=False)
        nb_virgules = masque_virgule.sum()
        if nb_virgules > 0:
            print(f"⚠️  {nb_virgules} valeurs avec virgule détectées dans '{colonne}'.")
            print("   → Remplacer les virgules par des points avant conversion.")
            print("   Exemple :", df.loc[masque_virgule, colonne].iloc[0])
        else:
            print(f"✅ Pas de virgule détectée dans '{colonne}'.")
    else:
        print(f"ℹ️  '{colonne}' n'est pas de type texte, pas de vérification nécessaire.")

df_original = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
verifier_format_numerique(df_original, "TotalCharges")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Forme du dataset :", df.shape)
print("\nTypes des colonnes :")
print(df.dtypes)
print("\nAperçu des 5 premières lignes :")
df.head()

In [ ]:
def audit_qualite(df):
    """
    Affiche un rapport de santé du dataset.
    Montre : dimensions, types, % manquants par colonne,
    et la répartition de la cible Churn.
    """
    print("=" * 55)
    print("         RAPPORT D'AUDIT QUALITÉ")
    print("=" * 55)

    nb_lignes, nb_colonnes = df.shape
    print(f"\n📐 Dimensions : {nb_lignes} lignes × {nb_colonnes} colonnes")

    print("\n📋 Types des colonnes :")
    print(df.dtypes.to_string())

    nb_manquants = df.isna().sum()
    pct_manquants = (df.isna().mean() * 100).round(2)
    tableau_manquants = pd.DataFrame({
        "manquants": nb_manquants,
        "pourcent": pct_manquants
    })
    colonnes_avec_trous = tableau_manquants[tableau_manquants["manquants"] > 0]

    print("\n🕳️  Valeurs manquantes :")
    if colonnes_avec_trous.empty:
        print("   Aucun NaN détecté (attention : des trous peuvent être cachés, voir Phase 2)")
    else:
        print(colonnes_avec_trous.sort_values("pourcent", ascending=False).to_string())

    nb_doublons = df.duplicated().sum()
    print(f"\n♻️  Doublons : {nb_doublons}")

    if "Churn" in df.columns:
        print("\n🎯 Répartition de la cible Churn :")
        counts = df["Churn"].value_counts()
        pcts = df["Churn"].value_counts(normalize=True) * 100
        for val in counts.index:
            print(f"   {val} : {counts[val]} ({pcts[val]:.1f}%)")
        # Alerte si déséquilibre
        ratio_min = pcts.min()
        if ratio_min < 30:
            print(f"   ⚠️  ATTENTION : cible déséquilibrée ({ratio_min:.1f}% pour la classe minoritaire)")
            print("      → L'accuracy seule sera trompeuse demain. Regarder aussi recall/F1.")
    else:
        print("\n⚠️  Colonne 'Churn' introuvable dans ce dataset.")

    print("\n" + "=" * 55)


audit_qualite(df)